## 🙏 Credits

# This notebook ensembles two excellent public BirdCLEF 2026 works: [Maryna Borovska](https://www.kaggle.com/marynaborovska)’s [ProtoSSM / Two-Pass SSM notebook](https://www.kaggle.com/code/marynaborovska/birdclef-26-two-pass-ssm-advanced-pp) and [Tucker Arrants](https://www.kaggle.com/tuckerarrants)’s [EfficientNet Distilled SED notebook](https://www.kaggle.com/code/tuckerarrants/bc2026-distilled-sed?scriptVersionId=314259833).

In [ ]:
# BirdCLEF 2026 — single-cell submit-only Perch + two-pass SSM + PP
import os, re, gc, time, warnings, subprocess, sys
from pathlib import Path

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")

INPUT_ROOT = Path("/kaggle/input")
BASE = Path("/kaggle/input/competitions/birdclef-2026")
ONNX_WHL = Path("/kaggle/input/datasets/rishikeshjani/perch-onnx-for-birdclef-2026/onnxruntime-1.24.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl")


def find_wheel(pattern):
    for p in INPUT_ROOT.rglob(pattern):
        return p
    raise FileNotFoundError(pattern)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", str(ONNX_WHL)], check=True)
    
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                str(find_wheel("tensorboard-2.20.0-*.whl"))], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                str(find_wheel("tensorflow-2.20.0-*.whl"))], check=True)

import onnxruntime as ort
import tensorflow as tf
import numpy as np
import pandas as pd
import soundfile as sf
from tqdm.auto import tqdm

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.isotonic import IsotonicRegression

import torch
import torch.nn as nn
import torch.nn.functional as F

tf.experimental.numpy.experimental_enable_numpy_behavior()

MODE = "submit"
WORK_DIR = Path("/kaggle/working/cache")
WORK_DIR.mkdir(parents=True, exist_ok=True)

SR = 32000
WINDOW_SEC = 5
WINDOW_SAMPLES = SR * WINDOW_SEC
FILE_SAMPLES = 60 * SR
N_WINDOWS = 12

CFG = {
    "batch_files": 16,
    "dryrun_n_files": 10,
    "verbose": False,
    "proto_epochs": 40,
    "proto_patience": 8,
    "res_epochs": 20,
    "res_patience": 6,
}

taxonomy = pd.read_csv(BASE / "taxonomy.csv")
sample_sub = pd.read_csv(BASE / "sample_submission.csv")
soundscape_labels = pd.read_csv(BASE / "train_soundscapes_labels.csv")

PRIMARY_LABELS = sample_sub.columns[1:].tolist()
N_CLASSES = len(PRIMARY_LABELS)
label_to_idx = {c: i for i, c in enumerate(PRIMARY_LABELS)}

FNAME_RE = re.compile(r"BC2026_(?:Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg")

def parse_fname(name):
    m = FNAME_RE.match(str(name))
    if not m:
        return {"site": "unknown", "hour_utc": -1}
    _, site, _, hms = m.groups()
    return {"site": site, "hour_utc": int(hms[:2])}

def union_labels(series):
    out = set()
    for x in series:
        if pd.notna(x):
            for t in str(x).split(";"):
                t = t.strip()
                if t:
                    out.add(t)
    return sorted(out)

sc = (
    soundscape_labels
    .groupby(["filename", "start", "end"])["primary_label"]
    .apply(union_labels)
    .reset_index(name="label_list")
)
sc["end_sec"] = pd.to_timedelta(sc["end"]).dt.total_seconds().astype(int)
sc["row_id"] = sc["filename"].str.replace(".ogg", "", regex=False) + "_" + sc["end_sec"].astype(str)
sc = pd.concat([sc, sc["filename"].apply(parse_fname).apply(pd.Series)], axis=1)

Y_SC = np.zeros((len(sc), N_CLASSES), dtype=np.uint8)
for i, lbls in enumerate(sc["label_list"]):
    for lbl in lbls:
        if lbl in label_to_idx:
            Y_SC[i, label_to_idx[lbl]] = 1

windows_per_file = sc.groupby("filename").size()
full_files = sorted(windows_per_file[windows_per_file == N_WINDOWS].index.tolist())
sc["fully_labeled"] = sc["filename"].isin(full_files)
full_rows = sc[sc["fully_labeled"]].sort_values(["filename", "end_sec"]).reset_index(drop=False)
Y_FULL = Y_SC[full_rows["index"].to_numpy()]

# Perch model / labels
MODEL_DIR = Path("/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1")
if not MODEL_DIR.exists():
    hits = sorted(INPUT_ROOT.glob("**/assets/labels.csv"))
    MODEL_DIR = hits[0].parents[1] if hits else MODEL_DIR

ONNX_PERCH_PATH = next(INPUT_ROOT.glob("**/perch_v2*.onnx"), Path(""))
USE_ONNX = ONNX_PERCH_PATH.exists()

if USE_ONNX:
    so = ort.SessionOptions()
    so.intra_op_num_threads = 4
    ONNX_SESSION = ort.InferenceSession(str(ONNX_PERCH_PATH), sess_options=so, providers=["CPUExecutionProvider"])
    ONNX_INPUT_NAME = ONNX_SESSION.get_inputs()[0].name
    ONNX_OUT_MAP = {o.name: i for i, o in enumerate(ONNX_SESSION.get_outputs())}
    print("Using ONNX Perch:", ONNX_PERCH_PATH)
else:
    birdclassifier = tf.saved_model.load(str(MODEL_DIR))
    infer_fn = birdclassifier.signatures["serving_default"]
    print("Using TF Perch:", MODEL_DIR)

bc_labels = (
    pd.read_csv(MODEL_DIR / "assets" / "labels.csv")
    .reset_index()
    .rename(columns={"index": "bc_index", "inat2024_fsd50k": "scientific_name"})
)
NO_LABEL = len(bc_labels)

mapping = taxonomy.merge(
    bc_labels.rename(columns={"scientific_name": "scientific_name"}),
    on="scientific_name",
    how="left",
)
mapping["bc_index"] = mapping["bc_index"].fillna(NO_LABEL).astype(int)
lbl2bc = mapping.set_index("primary_label")["bc_index"]

BC_INDICES = np.array([int(lbl2bc.loc[c]) for c in PRIMARY_LABELS], dtype=np.int32)
MAPPED_MASK = BC_INDICES != NO_LABEL
MAPPED_POS = np.where(MAPPED_MASK)[0].astype(np.int32)
MAPPED_BC_IDX = BC_INDICES[MAPPED_MASK].astype(np.int32)

UNMAPPED_POS = np.where(~MAPPED_MASK)[0].astype(np.int32)
CLASS_NAME_MAP = taxonomy.set_index("primary_label")["class_name"].to_dict()

proxy_map = {}
for _, row in taxonomy[taxonomy["primary_label"].isin([PRIMARY_LABELS[i] for i in UNMAPPED_POS])].iterrows():
    target = row["primary_label"]
    genus = str(row["scientific_name"]).split()[0]
    hits = bc_labels[bc_labels["scientific_name"].astype(str).str.match(rf"^{re.escape(genus)}\s", na=False)]
    if len(hits):
        proxy_map[label_to_idx[target]] = hits["bc_index"].astype(int).tolist()

PROXY_TAXA = {"Amphibia", "Insecta", "Aves"}
proxy_map = {
    idx: bc_idxs
    for idx, bc_idxs in proxy_map.items()
    if CLASS_NAME_MAP.get(PRIMARY_LABELS[idx]) in PROXY_TAXA
}

def read_60s(path):
    y, sr = sf.read(path, dtype="float32", always_2d=False)
    if y.ndim == 2:
        y = y.mean(axis=1)
    if len(y) < FILE_SAMPLES:
        y = np.pad(y, (0, FILE_SAMPLES - len(y)))
    else:
        y = y[:FILE_SAMPLES]
    return y.astype(np.float32, copy=False)

def run_perch(paths, batch_files=16, verbose=True):
    paths = [Path(p) for p in paths]
    n_rows = len(paths) * N_WINDOWS

    row_ids = np.empty(n_rows, dtype=object)
    filenames = np.empty(n_rows, dtype=object)
    sites = np.empty(n_rows, dtype=object)
    hours = np.zeros(n_rows, dtype=np.int16)
    scores = np.zeros((n_rows, N_CLASSES), dtype=np.float32)
    embs = np.zeros((n_rows, 1536), dtype=np.float32)

    wr = 0
    itr = range(0, len(paths), batch_files)
    if verbose:
        itr = tqdm(itr, desc="Perch")

    for start in itr:
        batch_paths = paths[start:start + batch_files]
        batch_audio = [read_60s(p) for p in batch_paths]
        x = np.empty((len(batch_paths) * N_WINDOWS, WINDOW_SAMPLES), dtype=np.float32)
        br = wr

        for bi, path in enumerate(batch_paths):
            y = batch_audio[bi]
            meta = parse_fname(path.name)
            stem = path.stem
            x[bi * N_WINDOWS:(bi + 1) * N_WINDOWS] = y.reshape(N_WINDOWS, WINDOW_SAMPLES)
            row_ids[wr:wr + N_WINDOWS] = [f"{stem}_{t}" for t in range(5, 65, 5)]
            filenames[wr:wr + N_WINDOWS] = path.name
            sites[wr:wr + N_WINDOWS] = meta["site"]
            hours[wr:wr + N_WINDOWS] = meta["hour_utc"]
            wr += N_WINDOWS

        if USE_ONNX:
            outs = ONNX_SESSION.run(None, {ONNX_INPUT_NAME: x})
            logits = outs[ONNX_OUT_MAP.get("label", 0)].astype(np.float32)
            emb = outs[ONNX_OUT_MAP.get("embedding", 1)].astype(np.float32)
        else:
            out = infer_fn(inputs=tf.convert_to_tensor(x))
            logits = out["label"].numpy().astype(np.float32)
            emb = out["embedding"].numpy().astype(np.float32)

        scores[br:wr, MAPPED_POS] = logits[:, MAPPED_BC_IDX]
        embs[br:wr] = emb

        for pos_idx, bc_idxs in proxy_map.items():
            scores[br:wr, pos_idx] = logits[:, np.asarray(bc_idxs, dtype=np.int32)].max(axis=1)

        del x, logits, emb, batch_audio
        gc.collect()

    meta_df = pd.DataFrame({"row_id": row_ids, "filename": filenames, "site": sites, "hour_utc": hours})
    return meta_df, scores, embs

# train cache
EXTERNAL_CACHE_DIRS = [p for p in INPUT_ROOT.glob("**") if p.is_dir()]
CACHE_META_LOCAL = WORK_DIR / "perch_meta.parquet"
CACHE_NPZ_LOCAL = WORK_DIR / "perch_arrays.npz"

SCORE_KEYS = ["scores", "sc", "scores_full_raw", "logits", "perch_scores", "preds", "arr_0"]
EMB_KEYS = ["embs", "emb", "emb_full", "embeddings", "features", "perch_embs", "arr_1"]

def _find_external_cache():
    pairs = [
        ("perch_meta.parquet", "perch_arrays.npz"),
        ("full_perch_meta.parquet", "full_perch_arrays.npz"),
    ]
    for d in EXTERNAL_CACHE_DIRS:
        for meta_name, npz_name in pairs:
            meta = d / meta_name
            npz = d / npz_name
            if meta.exists() and npz.exists():
                return meta, npz
    return None, None

def _pick_array(arr, candidates, shape_hint_cols):
    for k in candidates:
        if k in arr.files:
            return arr[k], k
    for k in arr.files:
        v = arr[k]
        if v.ndim == 2 and v.shape[1] == shape_hint_cols:
            return v, k
    raise KeyError(f"Could not find array for {shape_hint_cols}; keys={arr.files}")

def _build_cache():
    train_paths = [BASE / "train_soundscapes" / fn for fn in full_files]
    train_paths = [p for p in train_paths if p.exists()]
    meta_built, sc_built, emb_built = run_perch(train_paths, batch_files=CFG["batch_files"], verbose=True)
    meta_built.to_parquet(CACHE_META_LOCAL)
    np.savez(CACHE_NPZ_LOCAL, scores=sc_built.astype(np.float32), embs=emb_built.astype(np.float32), primary_labels=np.array(PRIMARY_LABELS))
    return CACHE_META_LOCAL, CACHE_NPZ_LOCAL

ext_meta, ext_npz = _find_external_cache()
if ext_meta is not None:
    CACHE_META, CACHE_NPZ = ext_meta, ext_npz
elif CACHE_META_LOCAL.exists() and CACHE_NPZ_LOCAL.exists():
    CACHE_META, CACHE_NPZ = CACHE_META_LOCAL, CACHE_NPZ_LOCAL
else:
    CACHE_META, CACHE_NPZ = _build_cache()

meta_tr = pd.read_parquet(CACHE_META)
arr = np.load(CACHE_NPZ)
sc_tr, sk = _pick_array(arr, SCORE_KEYS, N_CLASSES)
emb_tr, ek = _pick_array(arr, EMB_KEYS, 1536)
sc_tr = sc_tr.astype(np.float32)
emb_tr = emb_tr.astype(np.float32)

if "row_id" not in meta_tr.columns:
    if "end_sec" in meta_tr.columns:
        end_sec = meta_tr["end_sec"].astype(int)
    elif "window_idx" in meta_tr.columns:
        end_sec = (meta_tr["window_idx"].astype(int) + 1) * 5
    else:
        end_sec = np.tile(np.arange(5, 65, 5), len(meta_tr) // N_WINDOWS)
    meta_tr["row_id"] = meta_tr["filename"].str.replace(".ogg", "", regex=False) + "_" + end_sec.astype(str)

row_id_to_index = full_rows.set_index("row_id")["index"]
missing_rows = set(meta_tr["row_id"]) - set(row_id_to_index.index)
if missing_rows:
    raise RuntimeError(f"Train cache row_id mismatch. Example: {list(missing_rows)[:3]}")

Y_FULL_aligned = Y_SC[row_id_to_index.loc[meta_tr["row_id"]].to_numpy()].astype(np.uint8)

# postprocess
def temporal_smooth(probs, n_windows=12, alpha=0.25):
    out = probs.copy()
    v = probs.reshape(-1, n_windows, probs.shape[1])
    o = out.reshape(-1, n_windows, probs.shape[1])
    for t in range(n_windows):
        if t == 0:
            neigh = (v[:, t] + v[:, t + 1]) / 2
        elif t == n_windows - 1:
            neigh = (v[:, t - 1] + v[:, t]) / 2
        else:
            neigh = (v[:, t - 1] + v[:, t + 1]) / 2
        o[:, t] = (1 - alpha) * v[:, t] + alpha * neigh
    return out

def build_prior_tables(meta, Y, smooth_site=20, smooth_hour=30):
    df = meta.copy().reset_index(drop=True)
    df["site"] = df["site"].astype(str)
    df["hour_utc"] = df["hour_utc"].astype(int)
    global_logit = np.log((Y.mean(axis=0) + 1e-4) / (1 - Y.mean(axis=0) + 1e-4))
    site_prior, hour_prior = {}, {}
    for site, idx in df.groupby("site").indices.items():
        y = Y[list(idx)]
        p = (y.sum(axis=0) + smooth_site * Y.mean(axis=0)) / (len(idx) + smooth_site)
        site_prior[site] = np.log((p + 1e-4) / (1 - p + 1e-4))
    for hour, idx in df.groupby("hour_utc").indices.items():
        y = Y[list(idx)]
        p = (y.sum(axis=0) + smooth_hour * Y.mean(axis=0)) / (len(idx) + smooth_hour)
        hour_prior[int(hour)] = np.log((p + 1e-4) / (1 - p + 1e-4))
    return {"global": global_logit.astype(np.float32), "site": site_prior, "hour": hour_prior}

def apply_prior(scores, sites, hours, tables, lambda_prior=0.4):
    out = scores.copy()
    for i, (s, h) in enumerate(zip(sites, hours)):
        prior = tables["global"].copy()
        if str(s) in tables["site"]:
            prior = 0.5 * prior + 0.5 * tables["site"][str(s)]
        if int(h) in tables["hour"]:
            prior = 0.5 * prior + 0.5 * tables["hour"][int(h)]
        out[i] = out[i] + lambda_prior * prior
    return out

def file_confidence_scale(probs, n_windows=12, alpha=0.15):
    v = probs.reshape(-1, n_windows, probs.shape[1])
    top2 = np.sort(v, axis=1)[:, -2:].mean(axis=1, keepdims=True)
    return (v * ((1 - alpha) + alpha * top2)).reshape(probs.shape)

def rank_aware_scaling(probs, n_windows=12, power=0.4):
    v = probs.reshape(-1, n_windows, probs.shape[1])
    scale = np.power(v.max(axis=1, keepdims=True), power)
    return (v * scale).reshape(probs.shape)

def adaptive_delta_smooth(probs, n_windows=12, base_alpha=0.20):
    out = probs.copy()
    v = probs.reshape(-1, n_windows, probs.shape[1])
    o = out.reshape(-1, n_windows, probs.shape[1])
    for t in range(n_windows):
        conf = v[:, t].max(axis=-1, keepdims=True)
        alpha = base_alpha * (1.0 - conf)
        if t == 0:
            neigh = (v[:, t] + v[:, t + 1]) / 2
        elif t == n_windows - 1:
            neigh = (v[:, t - 1] + v[:, t]) / 2
        else:
            neigh = (v[:, t - 1] + v[:, t + 1]) / 2
        o[:, t] = (1 - alpha) * v[:, t] + alpha * neigh
    return out

TAXON_TEMPS = {"Aves": 0.90, "Amphibia": 1.10, "Insecta": 1.15, "Mammalia": 1.00, "Reptilia": 1.00}
CLASS_TEMPERATURES = np.ones(N_CLASSES, dtype=np.float32)
for i, lbl in enumerate(PRIMARY_LABELS):
    cls = CLASS_NAME_MAP.get(lbl, "")
    CLASS_TEMPERATURES[i] = TAXON_TEMPS.get(cls, 1.0)

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))

def apply_per_class_thresholds(scores, thresholds):
    scaled = scores.copy()
    for c in range(scores.shape[1]):
        t = thresholds[c]
        above = scores[:, c] > t
        scaled[above, c] = 0.5 + 0.5 * (scores[above, c] - t) / (1 - t + 1e-8)
        scaled[~above, c] = 0.5 * scores[~above, c] / (t + 1e-8)
    return np.clip(scaled, 0, 1)

def calibrate_and_optimize_thresholds(oof_probs, Y_FULL, threshold_grid=None, n_windows=12):
    if threshold_grid is None:
        threshold_grid = [0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]
    n_cls = oof_probs.shape[1]
    thresholds = np.full(n_cls, 0.5, dtype=np.float32)
    file_oof = oof_probs.reshape(-1, n_windows, n_cls).max(axis=1)
    file_y = Y_FULL.reshape(-1, n_windows, n_cls).max(axis=1)
    for c in range(n_cls):
        y_true, y_prob = file_y[:, c], file_oof[:, c]
        if y_true.sum() < 3:
            continue
        try:
            ir = IsotonicRegression(out_of_bounds="clip")
            ir.fit(y_prob, y_true)
            y_cal = ir.transform(y_prob)
        except Exception:
            y_cal = y_prob
        best_f1, best_t = 0.0, 0.5
        for t in threshold_grid:
            pred = (y_cal >= t).astype(int)
            tp = ((pred == 1) & (y_true == 1)).sum()
            fp = ((pred == 1) & (y_true == 0)).sum()
            fn = ((pred == 0) & (y_true == 1)).sum()
            prec = tp / (tp + fp + 1e-8)
            rec = tp / (tp + fn + 1e-8)
            f1 = 2 * prec * rec / (prec + rec + 1e-8)
            if f1 > best_f1:
                best_f1, best_t = f1, t
        thresholds[c] = best_t
    return thresholds

# MLP probes
def build_class_freq_weights(Y, cap=10.0):
    pos = Y.sum(axis=0).astype(np.float32) + 1.0
    freq = pos / Y.shape[0]
    w = 1.0 / (freq ** 0.5)
    w = np.clip(w, 1.0, cap)
    return (w / w.mean()).astype(np.float32)

def build_sequential_features(scores_col, n_windows=12):
    x = scores_col.reshape(-1, n_windows)
    prev = np.concatenate([x[:, :1], x[:, :-1]], axis=1)
    nxt = np.concatenate([x[:, 1:], x[:, -1:]], axis=1)
    mean = np.repeat(x.mean(axis=1), n_windows)
    mx = np.repeat(x.max(axis=1), n_windows)
    std = np.repeat(x.std(axis=1), n_windows)
    return prev.reshape(-1), nxt.reshape(-1), mean, mx, std

def train_mlp_probes(emb, scores_raw, Y, min_pos=5, pca_dim=128, alpha_blend=0.4):
    scaler = StandardScaler()
    emb_s = scaler.fit_transform(emb)
    pca = PCA(n_components=min(pca_dim, emb_s.shape[1] - 1))
    Z = pca.fit_transform(emb_s).astype(np.float32)

    cw = build_class_freq_weights(Y, cap=10.0)
    active = np.where(Y.sum(axis=0) >= min_pos)[0]
    probe_models = {}
    MAX_ROWS = 3000

    for ci in tqdm(active, desc="MLP probes"):
        y = Y[:, ci]
        if y.sum() == 0 or y.sum() == len(y):
            continue
        prev, nxt, mean, mx, std = build_sequential_features(scores_raw[:, ci])
        X = np.hstack([Z, scores_raw[:, ci:ci + 1], prev[:, None], nxt[:, None], mean[:, None], mx[:, None], std[:, None]])

        n_pos = int(y.sum())
        n_neg = len(y) - n_pos
        pos_idx = np.where(y == 1)[0]
        repeat = max(1, int(round(float(cw[ci]) * n_neg / max(n_pos, 1))))
        repeat = min(repeat, 8)
        if n_pos * repeat + len(y) > MAX_ROWS:
            repeat = max(1, (MAX_ROWS - len(y)) // max(n_pos, 1))

        X_bal = np.vstack([X, np.tile(X[pos_idx], (repeat, 1))])
        y_bal = np.concatenate([y, np.ones(n_pos * repeat, dtype=y.dtype)])

        clf = MLPClassifier(
            hidden_layer_sizes=(128, 64),
            activation="relu",
            max_iter=200,
            early_stopping=True,
            validation_fraction=0.15,
            n_iter_no_change=10,
            random_state=42,
            learning_rate_init=5e-4,
            alpha=0.005,
        )
        clf.fit(X_bal, y_bal)
        probe_models[ci] = clf

    return probe_models, scaler, pca, alpha_blend

class VectorizedMLPProbes(nn.Module):
    def __init__(self, probe_models):
        super().__init__()
        self.valid_classes = sorted(probe_models.keys())
        if not self.valid_classes:
            self.n_layers = 0
            self.weights = nn.ParameterList()
            self.biases = nn.ParameterList()
            return
        sample = probe_models[self.valid_classes[0]]
        self.n_layers = len(sample.coefs_)
        self.weights = nn.ParameterList()
        self.biases = nn.ParameterList()
        for li in range(self.n_layers):
            W = np.stack([probe_models[c].coefs_[li] for c in self.valid_classes], axis=0)
            b = np.stack([probe_models[c].intercepts_[li] for c in self.valid_classes], axis=0)
            self.weights.append(nn.Parameter(torch.tensor(W, dtype=torch.float32), requires_grad=False))
            self.biases.append(nn.Parameter(torch.tensor(b, dtype=torch.float32), requires_grad=False))

    def forward(self, x):
        h = x
        for i in range(self.n_layers):
            h = torch.bmm(h, self.weights[i]) + self.biases[i].unsqueeze(1)
            if i < self.n_layers - 1:
                h = torch.relu(h)
        return h.squeeze(-1)

def apply_mlp_probes_vectorized(emb_test, scores_test, probe_models, scaler, pca, alpha_blend=0.4):
    if len(probe_models) == 0:
        return scores_test.copy()

    emb_s = scaler.transform(emb_test)
    Z_test = pca.transform(emb_s).astype(np.float32)

    valid_classes = sorted(probe_models.keys())
    V = len(valid_classes)
    N = len(scores_test)

    raw = scores_test[:, valid_classes].T
    n_files = N // N_WINDOWS
    raw_view = raw.reshape(V, n_files, N_WINDOWS)

    prev = np.concatenate([raw_view[:, :, :1], raw_view[:, :, :-1]], axis=2).reshape(V, N)
    nxt = np.concatenate([raw_view[:, :, 1:], raw_view[:, :, -1:]], axis=2).reshape(V, N)
    mean = np.repeat(raw_view.mean(axis=2), N_WINDOWS, axis=1)
    mx = np.repeat(raw_view.max(axis=2), N_WINDOWS, axis=1)
    std = np.repeat(raw_view.std(axis=2), N_WINDOWS, axis=1)

    scalar_feats = np.stack([raw, prev, nxt, mean, mx, std], axis=-1).astype(np.float32)
    Z_expanded = np.broadcast_to(Z_test, (V, N, Z_test.shape[1]))
    X_all = np.concatenate([Z_expanded.astype(np.float32), scalar_feats], axis=-1)

    vec_probe = VectorizedMLPProbes(probe_models).eval()
    with torch.no_grad():
        preds = vec_probe(torch.tensor(X_all)).numpy()

    result = scores_test.copy()
    base_valid = scores_test[:, valid_classes]
    result[:, valid_classes] = (1.0 - alpha_blend) * base_valid + alpha_blend * preds.T
    return result

# SSMs
class SelectiveSSM(nn.Module):
    def __init__(self, d_model, d_state=16, d_conv=4):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.in_proj = nn.Linear(d_model, 2 * d_model, bias=False)
        self.conv1d = nn.Conv1d(d_model, d_model, d_conv, padding=d_conv - 1, groups=d_model)
        self.dt_proj = nn.Linear(d_model, d_model, bias=True)
        A = torch.arange(1, d_state + 1, dtype=torch.float32).unsqueeze(0).expand(d_model, -1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(d_model))
        self.B_proj = nn.Linear(d_model, d_state, bias=False)
        self.C_proj = nn.Linear(d_model, d_state, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B_sz, T, D = x.shape
        xz = self.in_proj(x)
        x_ssm, z = xz.chunk(2, dim=-1)
        x_conv = self.conv1d(x_ssm.transpose(1, 2))[:, :, :T].transpose(1, 2)
        x_conv = F.silu(x_conv)
        dt = F.softplus(self.dt_proj(x_conv))
        A = -torch.exp(self.A_log)
        Bv = self.B_proj(x_conv)
        Cv = self.C_proj(x_conv)
        h = torch.zeros(B_sz, D, self.d_state, device=x.device, dtype=x.dtype)
        ys = []
        for t in range(T):
            dA = torch.exp(A[None] * dt[:, t, :, None])
            dB = dt[:, t, :, None] * Bv[:, t, None, :]
            h = h * dA + x[:, t, :, None] * dB
            ys.append((h * Cv[:, t, None, :]).sum(-1))
        y = torch.stack(ys, dim=1)
        return y + x * self.D[None, None, :]

class LightProtoSSM(nn.Module):
    def __init__(
        self,
        d_input=1536,
        d_model=128,
        d_state=16,
        n_classes=234,
        n_windows=12,
        dropout=0.15,
        n_sites=20,
        meta_dim=16,
        use_cross_attn=True,
        cross_attn_heads=2,
    ):
        super().__init__()
        self.n_classes = n_classes
        self.n_windows = n_windows
        self.use_cross_attn = use_cross_attn
        self.input_proj = nn.Sequential(nn.Linear(d_input, d_model), nn.LayerNorm(d_model), nn.GELU(), nn.Dropout(dropout))
        self.pos_enc = nn.Parameter(torch.randn(1, n_windows, d_model) * 0.02)
        self.site_emb = nn.Embedding(n_sites, meta_dim)
        self.hour_emb = nn.Embedding(24, meta_dim)
        self.meta_proj = nn.Linear(2 * meta_dim, d_model)
        self.ssm_fwd = nn.ModuleList([SelectiveSSM(d_model, d_state) for _ in range(2)])
        self.ssm_bwd = nn.ModuleList([SelectiveSSM(d_model, d_state) for _ in range(2)])
        self.ssm_merge = nn.ModuleList([nn.Linear(2 * d_model, d_model) for _ in range(2)])
        self.ssm_norm = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(2)])
        self.drop = nn.Dropout(dropout)
        if use_cross_attn:
            self.cross_attn = nn.ModuleList([nn.MultiheadAttention(d_model, cross_attn_heads, dropout=dropout, batch_first=True) for _ in range(2)])
            self.cross_norm = nn.ModuleList([nn.LayerNorm(d_model) for _ in range(2)])
        self.prototypes = nn.Parameter(torch.randn(n_classes, d_model) * 0.02)
        self.proto_scale = nn.Parameter(torch.ones(n_classes))
        self.proto_bias = nn.Parameter(torch.zeros(n_classes))
        self.direct_head = nn.Linear(d_model, n_classes)
        self.base_fusion = nn.Parameter(torch.zeros(n_classes))

    def forward(self, emb, perch_scores, site_ids=None, hours=None):
        B, T, _ = emb.shape
        h = self.input_proj(emb) + self.pos_enc[:, :T]
        if site_ids is not None and hours is not None:
            meta = torch.cat([
                self.site_emb(site_ids.clamp(0, self.site_emb.num_embeddings - 1)),
                self.hour_emb(hours.clamp(0, 23)),
            ], dim=-1)
            h = h + self.meta_proj(meta).unsqueeze(1)
        for i in range(2):
            res = h
            hf = self.ssm_fwd[i](h)
            hb = self.ssm_bwd[i](h.flip(1)).flip(1)
            h = self.drop(self.ssm_merge[i](torch.cat([hf, hb], dim=-1)))
            h = self.ssm_norm[i](h + res)
            if self.use_cross_attn:
                res = h
                h_att, _ = self.cross_attn[i](h, h, h)
                h = self.cross_norm[i](res + self.drop(h_att))
        h_norm = F.normalize(h, dim=-1)
        p_norm = F.normalize(self.prototypes, dim=-1)
        proto_logits = torch.matmul(h_norm, p_norm.T) * self.proto_scale + self.proto_bias
        direct_logits = self.direct_head(h)
        alpha = torch.sigmoid(self.base_fusion)
        return 0.4 * proto_logits + 0.4 * direct_logits + 0.2 * perch_scores * alpha

class ResidualSSM(nn.Module):
    def __init__(self, d_input=1536, d_scores=234, d_model=128, d_state=16, n_classes=234, n_windows=12, dropout=0.1, n_sites=20, meta_dim=8):
        super().__init__()
        self.input_proj = nn.Sequential(nn.Linear(d_input + d_scores, d_model), nn.LayerNorm(d_model), nn.GELU(), nn.Dropout(dropout))
        self.site_emb = nn.Embedding(n_sites, meta_dim)
        self.hour_emb = nn.Embedding(24, meta_dim)
        self.meta_proj = nn.Linear(2 * meta_dim, d_model)
        self.pos_enc = nn.Parameter(torch.randn(1, n_windows, d_model) * 0.02)
        self.ssm_fwd = SelectiveSSM(d_model, d_state)
        self.ssm_bwd = SelectiveSSM(d_model, d_state)
        self.ssm_merge = nn.Linear(2 * d_model, d_model)
        self.ssm_norm = nn.LayerNorm(d_model)
        self.ssm_drop = nn.Dropout(dropout)
        self.output_head = nn.Linear(d_model, n_classes)
        nn.init.zeros_(self.output_head.weight)
        nn.init.zeros_(self.output_head.bias)

    def forward(self, emb, first_pass, site_ids=None, hours=None):
        B, T, _ = emb.shape
        h = self.input_proj(torch.cat([emb, first_pass], dim=-1)) + self.pos_enc[:, :T]
        if site_ids is not None and hours is not None:
            meta = self.meta_proj(torch.cat([
                self.site_emb(site_ids.clamp(0, self.site_emb.num_embeddings - 1)),
                self.hour_emb(hours.clamp(0, 23)),
            ], dim=-1))
            h = h + meta.unsqueeze(1)
        res = h
        hf = self.ssm_fwd(h)
        hb = self.ssm_bwd(h.flip(1)).flip(1)
        h = self.ssm_drop(self.ssm_merge(torch.cat([hf, hb], dim=-1)))
        h = self.ssm_norm(h + res)
        return self.output_head(h)

def train_light_proto_ssm(emb, sc_raw, Y, meta, n_epochs=40, patience=8, lr=1e-3, verbose=False):
    n_files = len(emb) // N_WINDOWS
    emb_f = emb.reshape(n_files, N_WINDOWS, -1)
    sc_f = sc_raw.reshape(n_files, N_WINDOWS, -1)
    Y_f = Y.reshape(n_files, N_WINDOWS, -1).astype(np.float32)

    sites = meta.drop_duplicates("filename")["site"].astype(str).tolist()
    site2i = {s: i for i, s in enumerate(sorted(set(sites)))}
    site_ids = np.array([site2i[s] for s in sites], dtype=np.int64)
    hour_ids = meta.drop_duplicates("filename")["hour_utc"].astype(int).to_numpy() % 24

    model = LightProtoSSM(n_classes=N_CLASSES, n_sites=max(20, len(site2i) + 1))
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)

    X = torch.tensor(emb_f, dtype=torch.float32)
    S = torch.tensor(sc_f, dtype=torch.float32)
    Yt = torch.tensor(Y_f, dtype=torch.float32)
    site_t = torch.tensor(site_ids, dtype=torch.long)
    hour_t = torch.tensor(hour_ids, dtype=torch.long)

    pos = Y_f.sum(axis=(0, 1))
    neg = Y_f.shape[0] * Y_f.shape[1] - pos
    pw = torch.tensor(np.clip(neg / np.maximum(pos, 1), 1.0, 25.0), dtype=torch.float32)

    best, wait, best_state = 1e9, 0, None
    for ep in range(n_epochs):
        model.train()
        opt.zero_grad()
        out = model(X, S, site_t, hour_t)
        loss = F.binary_cross_entropy_with_logits(out, Yt, pos_weight=pw)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        v = float(loss.detach())
        if v < best:
            best, wait = v, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    return model, site2i

def run_tta_proto(proto_model, emb_files, sc_files, site_t, hour_t, shifts=[0]):
    proto_model.eval()
    outs = []
    emb_t = torch.tensor(emb_files, dtype=torch.float32)
    sc_t = torch.tensor(sc_files, dtype=torch.float32)
    for shift in shifts:
        e = torch.roll(emb_t, shift, dims=1) if shift else emb_t
        s = torch.roll(sc_t, shift, dims=1) if shift else sc_t
        with torch.no_grad():
            out = proto_model(e, s, site_ids=site_t, hours=hour_t).numpy()
        if shift:
            out = np.roll(out, -shift, axis=1)
        outs.append(out)
    return np.mean(outs, axis=0)

def train_residual_ssm(emb_full, first_pass_flat, Y_full, site_ids, hour_ids, n_epochs=20, patience=6, lr=8e-4, correction_weight=0.35):
    n_files = len(emb_full) // N_WINDOWS
    emb_f = emb_full.reshape(n_files, N_WINDOWS, -1)
    fp_f = first_pass_flat.reshape(n_files, N_WINDOWS, -1)
    lab_f = Y_full.reshape(n_files, N_WINDOWS, -1).astype(np.float32)
    fp_prob = sigmoid(fp_f)
    residuals = lab_f - fp_prob

    model = ResidualSSM(n_classes=N_CLASSES)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)

    X = torch.tensor(emb_f, dtype=torch.float32)
    FP = torch.tensor(fp_f, dtype=torch.float32)
    R = torch.tensor(residuals, dtype=torch.float32)
    site_t = torch.tensor(site_ids, dtype=torch.long)
    hour_t = torch.tensor(hour_ids, dtype=torch.long)

    best, wait, best_state = 1e9, 0, None
    for ep in range(n_epochs):
        model.train()
        opt.zero_grad()
        pred = model(X, FP, site_t, hour_t)
        loss = F.mse_loss(pred, R)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        v = float(loss.detach())
        if v < best:
            best, wait = v, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    with torch.no_grad():
        correction = model(X, FP, site_t, hour_t).numpy()

    corrected = fp_f + correction_weight * correction
    return model, corrected.reshape(-1, N_CLASSES).astype(np.float32)

# test Perch
test_paths = sorted((BASE / "test_soundscapes").glob("*.ogg"))
if not test_paths:
    test_paths = sorted((BASE / "train_soundscapes").glob("*.ogg"))[:CFG["dryrun_n_files"]]
    print("debug fallback train files:", len(test_paths))
else:
    print("test files:", len(test_paths))

meta_te, sc_te, emb_te = run_perch(test_paths, CFG["batch_files"], verbose=False)

# pipeline
t0 = time.time()
proto_model, site2i_tr = train_light_proto_ssm(
    emb_tr, sc_tr, Y_FULL_aligned, meta_tr,
    n_epochs=CFG["proto_epochs"],
    patience=CFG["proto_patience"],
    lr=1e-3,
)

n_test_files = len(sc_te) // N_WINDOWS
emb_te_f = emb_te.reshape(n_test_files, N_WINDOWS, -1)
sc_te_f = sc_te.reshape(n_test_files, N_WINDOWS, -1)

test_fnames = meta_te.drop_duplicates("filename")["filename"].tolist()
test_site_ids = np.array([
    min(site2i_tr.get(meta_te.loc[meta_te["filename"] == fn, "site"].iloc[0], 0), 19)
    for fn in test_fnames
], dtype=np.int64)
test_hour_ids = np.array([
    int(meta_te.loc[meta_te["filename"] == fn, "hour_utc"].iloc[0]) % 24
    for fn in test_fnames
], dtype=np.int64)

with torch.no_grad():
    proto_out = proto_model(
        torch.tensor(emb_te_f, dtype=torch.float32),
        torch.tensor(sc_te_f, dtype=torch.float32),
        site_ids=torch.tensor(test_site_ids, dtype=torch.long),
        hours=torch.tensor(test_hour_ids, dtype=torch.long),
    ).numpy()
proto_scores_flat = proto_out.reshape(-1, N_CLASSES).astype(np.float32)

prior_tables = build_prior_tables(sc, Y_SC)
sc_te_adjusted = apply_prior(
    sc_te,
    sites=meta_te["site"].to_numpy(),
    hours=meta_te["hour_utc"].to_numpy(),
    tables=prior_tables,
    lambda_prior=0.4,
)

probe_models, emb_scaler, emb_pca, alpha_blend = train_mlp_probes(
    emb=emb_tr,
    scores_raw=sc_tr,
    Y=Y_FULL_aligned,
    min_pos=5,
    pca_dim=128,
    alpha_blend=0.4,
)

sc_te_adjusted = apply_mlp_probes_vectorized(
    emb_te,
    sc_te_adjusted,
    probe_models,
    emb_scaler,
    emb_pca,
    alpha_blend,
)

ENSEMBLE_W = 0.5
first_pass_flat = ENSEMBLE_W * proto_scores_flat + (1.0 - ENSEMBLE_W) * sc_te_adjusted

n_tr_files = len(sc_tr) // N_WINDOWS
emb_tr_f = emb_tr.reshape(n_tr_files, N_WINDOWS, -1)
sc_tr_f = sc_tr.reshape(n_tr_files, N_WINDOWS, -1)

tr_fnames = meta_tr.drop_duplicates("filename")["filename"].tolist()
tr_site_ids = np.array([
    min(site2i_tr.get(meta_tr.loc[meta_tr["filename"] == fn, "site"].iloc[0], 0), 19)
    for fn in tr_fnames
], dtype=np.int64)
tr_hour_ids = np.array([
    int(meta_tr.loc[meta_tr["filename"] == fn, "hour_utc"].iloc[0]) % 24
    for fn in tr_fnames
], dtype=np.int64)

proto_tr_out = run_tta_proto(
    proto_model,
    emb_tr_f,
    sc_tr_f,
    site_t=torch.tensor(tr_site_ids, dtype=torch.long),
    hour_t=torch.tensor(tr_hour_ids, dtype=torch.long),
    shifts=[0, 1, -1, 2, -2],
)
proto_tr_flat = proto_tr_out.reshape(-1, N_CLASSES).astype(np.float32)

sc_tr_prior = apply_prior(
    sc_tr,
    sites=meta_tr["site"].to_numpy(),
    hours=meta_tr["hour_utc"].to_numpy(),
    tables=prior_tables,
    lambda_prior=0.4,
)
sc_tr_mlp = apply_mlp_probes_vectorized(
    emb_tr,
    sc_tr_prior,
    probe_models,
    emb_scaler,
    emb_pca,
    alpha_blend,
)
first_pass_tr = ENSEMBLE_W * proto_tr_flat + (1.0 - ENSEMBLE_W) * sc_tr_mlp

PER_CLASS_THRESHOLDS = calibrate_and_optimize_thresholds(
    sigmoid(first_pass_tr),
    Y_FULL_aligned,
    threshold_grid=[0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70],
    n_windows=N_WINDOWS,
)

res_model, _ = train_residual_ssm(
    emb_tr,
    first_pass_tr,
    Y_FULL_aligned,
    tr_site_ids,
    tr_hour_ids,
    n_epochs=CFG["res_epochs"],
    patience=CFG["res_patience"],
    lr=8e-4,
    correction_weight=0.35,
)

with torch.no_grad():
    correction_te = res_model(
        torch.tensor(emb_te_f, dtype=torch.float32),
        torch.tensor(first_pass_flat.reshape(n_test_files, N_WINDOWS, -1), dtype=torch.float32),
        site_ids=torch.tensor(test_site_ids, dtype=torch.long),
        hours=torch.tensor(test_hour_ids, dtype=torch.long),
    ).numpy().reshape(-1, N_CLASSES)

final_logits = first_pass_flat + 0.35 * correction_te
final_logits = final_logits / CLASS_TEMPERATURES[None, :]

probs = sigmoid(final_logits)
probs = file_confidence_scale(probs, n_windows=N_WINDOWS, alpha=0.15)
probs = rank_aware_scaling(probs, n_windows=N_WINDOWS, power=0.4)
probs = adaptive_delta_smooth(probs, n_windows=N_WINDOWS, base_alpha=0.20)
probs = apply_per_class_thresholds(probs, PER_CLASS_THRESHOLDS)
probs = np.clip(probs, 0.0, 1.0).astype(np.float32)

sub = pd.DataFrame(probs, columns=PRIMARY_LABELS)
sub.insert(0, "row_id", meta_te["row_id"].to_numpy())

if len(sample_sub) == len(sub) and set(sample_sub["row_id"]) == set(sub["row_id"]):
    sub = sub.set_index("row_id").loc[sample_sub["row_id"]].reset_index()

sub.to_csv("submission_protossm.csv", index=False)
protossm_sub = sub.copy()
print("saved submission_protossm.csv", sub.shape, f"time={time.time() - t0:.1f}s")
sub.head()

In [ ]:
# Cell 2 — distilled SED inference -> submission_sed.csv

# ---- original cell 1 ----
# Submit-only distilled SED ONNX inference

import os, sys, glob, re, time, subprocess
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter1d

try:
    import onnxruntime as ort
except Exception:
    wheels = sorted(Path("/kaggle/input").glob("**/onnxruntime*.whl"))
    if not wheels:
        wheels = [Path("/kaggle/input/datasets/tuckerarrants/perch-v2-no-dft-onnx/onnxruntime-1.24.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl")]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", str(wheels[0])], check=True)
    import onnxruntime as ort

import librosa

try:
    import soundfile as sf
except Exception:
    sf = None


COMP_DIR = Path("/kaggle/input/competitions/birdclef-2026")
if not COMP_DIR.exists():
    COMP_DIR = Path("/kaggle/input/birdclef-2026")

SAMPLE_SUB_PATH = COMP_DIR / "sample_submission.csv"
TEST_DIR = COMP_DIR / "test_soundscapes"

PRIMARY_LABELS = pd.read_csv(SAMPLE_SUB_PATH).columns[1:].tolist()
NUM_CLASSES = len(PRIMARY_LABELS)

SR = 32000
CHUNK_SEC = 5
CHUNK_N = SR * CHUNK_SEC
FILE_SEC = 60

N_MELS = 256
N_FFT = 2048
HOP = 512
FMIN = 20
FMAX = 16000
TOP_DB = 80


def find_sed_dir():
    candidates = [
        Path("/kaggle/input/datasets/tuckerarrants/bc2026-distilled-sed-public"),
        Path("/kaggle/input/bc2026-distilled-sed-public"),
        Path("/kaggle/input/bc2026-distilled-sed"),
        Path("/kaggle/working"),
    ]
    for d in candidates:
        if d.exists() and list(d.glob("sed_fold*.onnx")):
            return d
    hits = sorted(Path("/kaggle/input").glob("**/sed_fold*.onnx"))
    if not hits:
        raise FileNotFoundError("No sed_fold*.onnx found under /kaggle/input")
    return hits[0].parent


def make_session(path):
    so = ort.SessionOptions()
    so.intra_op_num_threads = 4
    so.inter_op_num_threads = 1
    so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
    avail = ort.get_available_providers()
    providers = [p for p in ["CUDAExecutionProvider", "CPUExecutionProvider"] if p in avail]
    return ort.InferenceSession(str(path), sess_options=so, providers=providers)


def audio_to_mel(chunks):
    mels = []
    for x in chunks:
        s = librosa.feature.melspectrogram(
            y=x, sr=SR, n_fft=N_FFT, hop_length=HOP,
            n_mels=N_MELS, fmin=FMIN, fmax=FMAX, power=2.0,
        )
        s = librosa.power_to_db(s, top_db=TOP_DB)
        s = (s - s.mean()) / (s.std() + 1e-6)
        mels.append(s)
    return np.stack(mels)[:, None].astype(np.float32)


def load_audio(path):
    if sf is not None:
        wav, sr = sf.read(path, dtype="float32", always_2d=False)
        if wav.ndim > 1:
            wav = wav.mean(axis=1)
        if sr != SR:
            wav = librosa.resample(wav, orig_sr=sr, target_sr=SR)
    else:
        wav, _ = librosa.load(path, sr=SR, mono=True)
    return wav.astype(np.float32)


def file_to_chunks(path):
    wav = load_audio(path)
    n = FILE_SEC * SR
    if len(wav) < n:
        wav = np.pad(wav, (0, n - len(wav)))
    else:
        wav = wav[:n]
    chunks = wav.reshape(FILE_SEC // CHUNK_SEC, CHUNK_N)
    ends = np.arange(1, chunks.shape[0] + 1) * CHUNK_SEC
    return chunks, ends


def sigmoid(x):
    x = np.clip(x, -50, 50)
    return (1.0 / (1.0 + np.exp(-x))).astype(np.float32)


sed_dir = find_sed_dir()
fold_paths = sorted(sed_dir.glob("sed_fold*.onnx"), key=lambda p: int(re.search(r"sed_fold(\d+)", p.name).group(1)))
sessions = [make_session(p) for p in fold_paths]

print("SED dir:", sed_dir)
print("folds:", [p.name for p in fold_paths])

test_files = sorted(TEST_DIR.glob("*.ogg"))
if not test_files:
    # Save-version/mock mode: force SED to use the exact same mock files as ProtoSSM.
    mock_files = []

    if "protossm_sub" in globals():
        proto_rows = protossm_sub["row_id"].astype(str)
    elif Path("submission_protossm.csv").exists():
        proto_rows = pd.read_csv("submission_protossm.csv")["row_id"].astype(str)
    else:
        proto_rows = None

    if proto_rows is not None:
        stems = proto_rows.str.rsplit("_", n=1).str[0].drop_duplicates().tolist()
        mock_files = [COMP_DIR / "train_soundscapes" / f"{s}.ogg" for s in stems]
        mock_files = [p for p in mock_files if p.exists()]

    test_files = mock_files if len(mock_files) else sorted((COMP_DIR / "train_soundscapes").glob("*.ogg"))[:10]
    print("debug fallback train files:", len(test_files))

rows, preds = [], []
t0 = time.time()

for i, path in enumerate(test_files, 1):
    chunks, ends = file_to_chunks(path)
    mel = audio_to_mel(chunks)
    p_sum = np.zeros((len(chunks), NUM_CLASSES), dtype=np.float32)

    for sess in sessions:
        outs = sess.run(None, {sess.get_inputs()[0].name: mel})
        clip_logits = outs[0]
        frame_max = outs[1].max(axis=1)
        p_sum += 0.5 * sigmoid(clip_logits) + 0.5 * sigmoid(frame_max)

    p_mean = p_sum / len(sessions)
    if len(p_mean) > 1:
        p_mean = gaussian_filter1d(p_mean, sigma=0.65, axis=0, mode="nearest").astype(np.float32)

    stem = path.stem
    rows.extend([f"{stem}_{int(t)}" for t in ends])
    preds.append(p_mean)

    if i == 1 or i == len(test_files) or i % 50 == 0:
        print(f"{i}/{len(test_files)} files | {time.time() - t0:.1f}s")

preds = np.concatenate(preds, axis=0) if preds else np.zeros((0, NUM_CLASSES), dtype=np.float32)

submission = pd.DataFrame(preds, columns=PRIMARY_LABELS)
submission.insert(0, "row_id", rows)
submission.iloc[:, 1:] = submission.iloc[:, 1:].clip(0.0, 1.0)
submission.to_csv("submission_sed.csv", index=False)

print("saved submission_sed.csv:", submission.shape)
submission.head()

sed_sub = submission.copy()
print("captured sed_sub:", sed_sub.shape)

In [ ]:
# Cell 3 — ensemble

PROTOSSM_W = 0.6
SED_W = 0.4
EPS = 1e-5

a = pd.read_csv("submission_protossm.csv")
b = pd.read_csv("submission_sed.csv")

cols = [c for c in a.columns if c != "row_id"]
missing = sorted(set(a["row_id"]) - set(b["row_id"]))
if missing:
    raise ValueError(f"row_id mismatch: {len(missing)} rows in ProtoSSM not in SED; first={missing[:3]}")
b = b.set_index("row_id").loc[a["row_id"]].reset_index()

pa = np.clip(a[cols].to_numpy("float32"), EPS, 1.0 - EPS)
pb = np.clip(b[cols].to_numpy("float32"), EPS, 1.0 - EPS)

wa = PROTOSSM_W / (PROTOSSM_W + SED_W)
wb = SED_W / (PROTOSSM_W + SED_W)

ra = pd.DataFrame(pa).rank(axis=0, pct=True).to_numpy("float32")
rb = pd.DataFrame(pb).rank(axis=0, pct=True).to_numpy("float32")
pred = wa * ra + wb * rb

submission = a.copy()
submission[cols] = pred.astype("float32")
submission.to_csv("submission.csv", index=False)

print("saved submission.csv", submission.shape, "weights", {"protossm": wa, "sed": wb})